# Clase 192 — Bayes: priors, posterior, MCMC con PyMC

Lógica bayesiana: `posterior ∝ likelihood × prior`. Mostramos el caso conjugado **Beta-Binomial** de forma ejecutable con scipy, y el stack moderno **PyMC v5 + ArviZ** en código correcto (no se ejecuta aquí porque PyMC no está instalado en este entorno).

Requiere ejecutar el conjugado: `numpy`, `scipy`, `matplotlib`. Para los modelos MCMC: `pip install pymc arviz numpyro`.

## 🧠 Intuición previa

**Bayes en una frase:** en vez de un p-value (¿qué tan raros serían mis datos si H₀ fuera cierta?), obtenés una **distribución de creencia sobre el parámetro** — `P(θ | datos)` — que **se actualiza con los datos** vía `posterior ∝ likelihood × prior`. Con eso respondés directo `P(θ > 0.5 | datos)`, sin la pirueta interpretativa del IC frecuentista.

## 1. Conjugado Beta-Binomial (ejecutable)

Con prior `Beta(1,1)` (uniforme) y datos binomiales, el posterior es `Beta(a+éxitos, b+fracasos)` — sin necesidad de MCMC. Calculamos el intervalo del 94 % con `scipy`.

In [ ]:
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# 100 visitas, 8 conversiones. Prior Beta(1,1)
visitas, conversiones = 100, 8
a0, b0 = 1, 1
a_post, b_post = a0 + conversiones, b0 + (visitas - conversiones)   # Beta(9, 93)
post = stats.beta(a_post, b_post)
lo, hi = post.ppf(0.03), post.ppf(0.97)    # intervalo central del 94%
print(f"Posterior = Beta({a_post}, {b_post})")
print(f"media posterior = {post.mean():.3f}")
print(f"intervalo 94% = ({lo:.3f}, {hi:.3f})")

grid = np.linspace(0, 0.25, 300)
plt.figure(figsize=(7, 4))
plt.plot(grid, stats.beta(a0, b0).pdf(grid), label="prior Beta(1,1)")
plt.plot(grid, post.pdf(grid), label=f"posterior Beta({a_post},{b_post})")
plt.title("Actualización bayesiana: prior -> posterior"); plt.legend()
plt.tight_layout(); plt.show()

## 2. Regresión lineal bayesiana con PyMC v5

Código moderno (PyTensor backend). NUTS muestrea el posterior; ArviZ diagnostica convergencia (`r_hat ≤ 1.01`, `ess_bulk ≥ 400`). **No se ejecuta aquí** (PyMC ausente).

In [ ]:
# Requiere: pip install pymc arviz  (NO se ejecuta en este entorno)
import numpy as np
import pymc as pm
import arviz as az

rng = np.random.default_rng(42)
x = rng.normal(0, 1, 200)
y = 1.0 + 2.5 * x + rng.normal(0, 1, 200)

with pm.Model() as modelo:
    alpha = pm.Normal("alpha", mu=0, sigma=10)          # priors débilmente informativos
    beta = pm.Normal("beta", mu=0, sigma=10)
    sigma = pm.HalfNormal("sigma", sigma=5)
    mu = alpha + beta * x
    y_obs = pm.Normal("y_obs", mu=mu, sigma=sigma, observed=y)
    idata = pm.sample(2000, tune=1000, chains=4, target_accept=0.9, random_seed=42)

print(az.summary(idata, var_names=["alpha", "beta", "sigma"]))
az.plot_trace(idata, var_names=["alpha", "beta"])

# Posterior predictive check: validación bayesiana fundamental
with modelo:
    ppc = pm.sample_posterior_predictive(idata, random_seed=42)
az.plot_ppc(ppc)

## 3. Modelo jerárquico (efectos por grupo)

`tip ~ Normal(α_día + β·total_bill, σ)` con `α_día ~ Normal(μ_α, σ_α)`. La jerarquía comparte información entre grupos (partial pooling), mejor que 4 regresiones separadas. Código correcto, **no ejecutado**.

In [ ]:
# Modelo jerárquico en PyMC v5 (ilustrativo, NO se ejecuta aquí)
import numpy as np
import pymc as pm
import arviz as az

rng = np.random.default_rng(0)
dia_idx = rng.integers(0, 4, 244)              # 4 días
total_bill = rng.gamma(4, 5, 244)
tip = 1.0 + 0.15 * total_bill + rng.normal(0, 1, 244)

with pm.Model() as jer:
    mu_a = pm.Normal("mu_a", 0, 5)
    sigma_a = pm.HalfNormal("sigma_a", 2)
    alpha_dia = pm.Normal("alpha_dia", mu_a, sigma_a, shape=4)   # intercepto por día
    beta = pm.Normal("beta", 0, 1)
    sigma = pm.HalfNormal("sigma", 2)
    mu = alpha_dia[dia_idx] + beta * total_bill
    pm.Normal("tip_obs", mu=mu, sigma=sigma, observed=tip)
    idata_jer = pm.sample(2000, tune=1000, chains=4, random_seed=0)

print(az.summary(idata_jer, var_names=["beta", "alpha_dia"]))
# HDI de β: si no incluye 0, el efecto del bill sobre la propina es claro
print(az.hdi(idata_jer, var_names=["beta"], hdi_prob=0.94))

## Ejercicios

1. En el conjugado, cambiá el prior a `Beta(2, 20)` (creencia previa de tasa baja) y observá cómo se desplaza el posterior con los mismos datos.
2. Ejecutá (en un entorno con PyMC) la regresión del bloque 2 y compará `mean` del posterior de `beta` contra el coeficiente OLS de `statsmodels`: con priors débiles deben coincidir.
3. Hacé un `pm.sample_prior_predictive` y verificá que los priors no generan datos absurdos (prior predictive check).

## Conclusiones

- El teorema de Bayes combina prior + likelihood en un posterior; en casos conjugados (Beta-Binomial) es analítico.
- El **HDI** sí tiene interpretación directa de probabilidad, a diferencia del IC frecuentista.
- PyMC v5 (NUTS) muestrea posteriores complejos; diagnosticá con `r_hat ≤ 1.01` y `ess_bulk ≥ 400`.
- El posterior predictive check es la validación bayesiana clave; los modelos jerárquicos comparten información entre grupos.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y **ejecutables** de los ejercicios del README (sección `## 🧪 Ejercicios`). Datos sintéticos con `np.random.default_rng(42)`, sin internet. Cada bloque imprime resultados y valida con `assert`.

> **Nota (PyMC):** PyMC/NumPyro/ArviZ no están instalados en este entorno. Los bloques con esas librerías son **código correcto ilustrativo** (guardado en `try/except`, no rompe la ejecución); la lógica bayesiana se valida con versiones **conjugadas/analíticas** en numpy/scipy que **sí** corren con `assert`.

### Ejercicio 1 — Conjugado a mano
100 visitas, 8 conversiones, prior `Beta(1,1)` → posterior `Beta(9, 93)`. HDI 94% con `scipy.stats.beta.ppf`.

In [ ]:
import numpy as np
from scipy import stats
visitas, conversiones = 100, 8
a0, b0 = 1, 1                                   # prior Beta(1,1)
a_post, b_post = a0 + conversiones, b0 + (visitas - conversiones)   # Beta(9, 93)
post = stats.beta(a_post, b_post)
central = (post.ppf(0.03), post.ppf(0.97))       # intervalo de credibilidad 94%
print(f"Posterior = Beta({a_post}, {b_post})  media = {post.mean():.3f}")
print(f"IC/HDI 94% = ({central[0]:.3f}, {central[1]:.3f})")
assert (a_post, b_post) == (9, 93)
assert central[0] < 0.08 < central[1]

### Ejercicio 2 — Regresión bayesiana
Código **PyMC v5 correcto** (ilustrativo) + versión **ejecutable**: regresión lineal bayesiana conjugada (prior Normal, σ conocida → posterior Normal).

In [ ]:
# --- PyMC v5 (correcto; corre solo si PyMC esta instalado) ---
try:
    import pymc as pm, arviz as az
    rng = np.random.default_rng(42)
    xg = rng.normal(0, 1, 200); yg = 1.0 + 2.5*xg + rng.normal(0, 1, 200)
    with pm.Model() as modelo:
        alpha = pm.Normal("alpha", 0, 10)
        beta = pm.Normal("beta", 0, 10)
        sigma = pm.HalfNormal("sigma", 5)
        pm.Normal("y_obs", mu=alpha + beta*xg, sigma=sigma, observed=yg)
        idata = pm.sample(2000, tune=1000, chains=4, target_accept=0.9, random_seed=42)
    print(az.summary(idata, var_names=["alpha", "beta", "sigma"]))
except Exception as e:
    print(f"PyMC no disponible ({type(e).__name__}); ver version analitica ejecutable abajo.")

In [ ]:
# --- Version ejecutable: regresion lineal bayesiana conjugada (sigma conocida) ---
rng = np.random.default_rng(42)
n = 200
x = rng.normal(0, 1, n)
sigma = 1.0
y = 1.0 + 2.5*x + rng.normal(0, sigma, n)        # alpha=1, beta=2.5
X = np.column_stack([np.ones(n), x])
tau2 = 100.0                                      # prior Normal(0, tau2 I) sobre [alpha, beta]
prec_post = np.eye(2)/tau2 + X.T @ X / sigma**2
cov_post = np.linalg.inv(prec_post)
mean_post = cov_post @ (X.T @ y / sigma**2)
sd_post = np.sqrt(np.diag(cov_post))
z = stats.norm.ppf(0.97)
hdi_b = (mean_post[1] - z*sd_post[1], mean_post[1] + z*sd_post[1])
print(f"posterior alpha = {mean_post[0]:.3f} +/- {sd_post[0]:.3f}")
print(f"posterior beta  = {mean_post[1]:.3f} +/- {sd_post[1]:.3f}")
print(f"HDI 94% beta = ({hdi_b[0]:.3f}, {hdi_b[1]:.3f})")
assert abs(mean_post[1] - 2.5) < 0.3 and hdi_b[0] > 0

### Ejercicio 3 — Comparación con OLS
Con priors débiles, la media del posterior de `beta` ≈ coeficiente OLS de `statsmodels`.

In [ ]:
import statsmodels.api as sm
ols = sm.OLS(y, X).fit()
print(f"OLS   : alpha={ols.params[0]:.3f}  beta={ols.params[1]:.3f}")
print(f"Bayes : alpha={mean_post[0]:.3f}  beta={mean_post[1]:.3f}")
assert abs(ols.params[1] - mean_post[1]) < 0.05

### Ejercicio 4 — Posterior predictive check
Código **ArviZ/PyMC** ilustrativo + PPC **ejecutable**: simular `y_rep` desde el posterior y comparar estadísticos con los observados (bayesian p-value ≈ 0.5).

In [ ]:
# --- PPC con PyMC/ArviZ (ilustrativo) ---
try:
    import pymc as pm, arviz as az
    with modelo:
        ppc = pm.sample_posterior_predictive(idata, random_seed=42)
    az.plot_ppc(ppc)
except Exception as e:
    print(f"PyMC/ArviZ no disponible ({type(e).__name__}); PPC analitico ejecutable abajo.")

# --- PPC ejecutable desde el posterior conjugado ---
draws = 4000
L = np.linalg.cholesky(cov_post)
params = mean_post + (L @ rng.normal(0, 1, (2, draws))).T          # muestras del posterior
y_rep = params @ X.T + rng.normal(0, sigma, (draws, n))            # datos replicados
p_mean = np.mean(y_rep.mean(axis=1) >= y.mean())
p_sd = np.mean(y_rep.std(axis=1) >= y.std())
print(f"bayesian p-value (media) = {p_mean:.2f}  (bien calibrado ~0.5)")
print(f"bayesian p-value (sd)    = {p_sd:.2f}")
assert 0.05 < p_mean < 0.95

### Ejercicio 5 — NumPyro
Traducción del modelo a **NumPyro** (JAX). Código correcto ilustrativo; corre solo en un entorno con JAX/NumPyro.

In [ ]:
try:
    import numpyro, numpyro.distributions as dist
    from numpyro.infer import MCMC, NUTS
    import jax.numpy as jnp
    from jax import random as jr
    def model(x, y=None):
        a = numpyro.sample("alpha", dist.Normal(0, 10))
        b = numpyro.sample("beta", dist.Normal(0, 10))
        s = numpyro.sample("sigma", dist.HalfNormal(5))
        numpyro.sample("y_obs", dist.Normal(a + b*x, s), obs=y)
    mcmc = MCMC(NUTS(model), num_warmup=500, num_samples=1000, progress_bar=False)
    mcmc.run(jr.PRNGKey(42), x=jnp.array(x), y=jnp.array(y))
    mcmc.print_summary()
except Exception as e:
    print(f"NumPyro no disponible ({type(e).__name__}); codigo correcto para entorno con JAX.")